In [1]:
# CÉLULA 1: Configuração
# O notebook está dentro da pasta SRC-2nd
# e os ficheiros db-ip estão na mesma pasta do notebook.

BASE_PATH = './'
DATASET_PATH = './dataset10/'

# Ficheiros de treino
INTERNAL_TRAIN = DATASET_PATH + 'internal_train10.json'
EXTERNAL_TRAIN = DATASET_PATH + 'external_train10.json'

# Ficheiros de teste
INTERNAL_TEST  = DATASET_PATH + 'internal_test10.json'
EXTERNAL_TEST  = DATASET_PATH + 'external_test10.json'

# Bases de dados db-ip
GEODB_COUNTRY_PATH = BASE_PATH + 'dbip-country-lite-2026-05.mmdb'
GEODB_ASN_PATH     = BASE_PATH + 'dbip-asn-lite-2026-05.mmdb'

# Redes
CORP_NET_STR = '200.0.0.0/24'
INTERNAL_NET_STR = '192.168.110.0/24'

print('✅ Ficheiros  carregados!')
print(f'   internal_train : {INTERNAL_TRAIN}')
print(f'   external_train : {EXTERNAL_TRAIN}')
print(f'   geodb country  : {GEODB_COUNTRY_PATH}')
print(f'   geodb asn      : {GEODB_ASN_PATH}')

✅ Ficheiros  carregados!
   internal_train : ./dataset10/internal_train10.json
   external_train : ./dataset10/external_train10.json
   geodb country  : ./dbip-country-lite-2026-05.mmdb
   geodb asn      : ./dbip-asn-lite-2026-05.mmdb


In [2]:
# CÉLULA 2: Configuração Bibliotecas e geolocalização
import pandas as pd
import numpy as np
import ipaddress
import geoip2.database
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# Melhora a qualidade visual dos gráficos
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi']    = 100

# Abre as bases de dados de geolocalização db-ip
# geodb    → converte IP em país (ex: PT, US, CN)
# geodbasn → converte IP em organização/operador (ex: Google LLC)
geodb    = geoip2.database.Reader(GEODB_COUNTRY_PATH)
geodbasn = geoip2.database.Reader(GEODB_ASN_PATH)

# Converte as strings de rede em objetos IPv4Network para comparações
CORP_NET     = ipaddress.IPv4Network(CORP_NET_STR)
INTERNAL_NET = ipaddress.IPv4Network(INTERNAL_NET_STR)

# Retorna o código ISO do país de um IP público (ex: 'PT', 'US')
# Se falhar (IP privado ou não encontrado) devolve 'PRIVATE'
def get_country(ip):
    try:    return geodb.country(ip).country.iso_code
    except: return 'PRIVATE'

# Retorna o nome da organização/ASN de um IP
def get_asn(ip):
    try:    return geodbasn.asn(ip).autonomous_system_organization
    except: return 'UNKNOWN'

# Verifica se um IP é privado (192.168.x.x, 10.x.x.x, 172.16.x.x, etc.)
def is_private(ip):
    try:    return ipaddress.IPv4Address(ip).is_private
    except: return False

print('✅ Bibliotecas e geolocalização carregadas!')

✅ Bibliotecas e geolocalização carregadas!


In [3]:
# Task 1 - Caracterização do Tráfego de Referência
# Nesta secção carregamos os datasets de treino e construímos o baseline de comportamento
# dos clientes internos, que servirá de referência para a definição das regras UEBA.

In [4]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 3 — Carregamento dos dados de treino
# Lê internal_train10 e external_train10 em DataFrames pandas
# ═══════════════════════════════════════════════════════════════

print('A carregar internal_train10...')
internal_train = pd.read_json(INTERNAL_TRAIN)
print(f'internal_train10: {len(internal_train):,} flows')

print('\nA carregar external_train10...')
external_train = pd.read_json(EXTERNAL_TRAIN)
print(f'external_train10: {len(external_train):,} flows')

print(f'\nColunas disponíveis: {list(internal_train.columns)}')

print('\n--- Amostra internal_train10 ---')
display(internal_train.sample(5))

print('\n--- Amostra external_train10 ---')
display(external_train.sample(5))

A carregar internal_train10...
internal_train10: 963,503 flows

A carregar external_train10...
external_train10: 685,960 flows

Colunas disponíveis: ['timestamp', 'src_ip', 'dst_ip', 'proto', 'port', 'up_bytes', 'down_bytes']

--- Amostra internal_train10 ---


,timestamp,src_ip,dst_ip,proto,port,up_bytes,down_bytes
411532,4720533,192.168.110.137,142.251.154.119,tcp,443,11757,66352
308267,4227345,192.168.110.206,192.168.110.230,udp,53,208,490
31651,4937392,192.168.110.73,192.168.110.238,tcp,443,6050,42086
481926,4320991,192.168.110.197,88.157.217.145,tcp,443,10161,73146
550769,7345610,192.168.110.118,142.251.154.119,tcp,443,15432,78597



--- Amostra external_train10 ---


,timestamp,src_ip,dst_ip,proto,port,up_bytes,down_bytes
446463,6118030,188.83.81.164,200.0.0.12,tcp,443,11231,69686
295624,7711404,188.83.81.103,200.0.0.11,tcp,443,6313,49078
50185,3823816,188.83.81.181,200.0.0.12,tcp,443,9879,56294
568905,2461961,188.83.81.187,200.0.0.12,tcp,443,15367,112009
553260,4349913,188.83.81.50,200.0.0.11,tcp,443,14349,132913


In [5]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 3b — Identificar a rede privada interna observada
# ═══════════════════════════════════════════════════════════════
all_ips     = pd.concat([internal_train['src_ip'], internal_train['dst_ip']]).unique()
private_ips = [ip for ip in all_ips if is_private(ip)]

nets = set()
for ip in private_ips:
    addr = ipaddress.IPv4Address(ip)
    net  = ipaddress.IPv4Network(f'{addr.packed[0]}.{addr.packed[1]}.{addr.packed[2]}.0/24')
    nets.add(str(net))

print('Redes privadas observadas no internal_train:')
for n in sorted(nets):
    print(f'  {n}')

Redes privadas observadas no internal_train:
  192.168.110.0/24


In [6]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 4 — Task 1: Baselines internos
# Agrega métricas por IP interno (volume, destinos, intervals, DNS, países)
# ═══════════════════════════════════════════════════════════════

# ── 1) Baseline de clientes internos ──────────────────────────
baseline_clients = internal_train.groupby('src_ip').agg(
    n_flows      = ('up_bytes',   'count'),
    total_up     = ('up_bytes',   'sum'),
    total_down   = ('down_bytes', 'sum'),
    n_dst_servers= ('dst_ip',     'nunique'),
    n_ports      = ('port',       'nunique')
).reset_index()

baseline_clients['ratio_up_down'] = (
    baseline_clients['total_up'] /
    (baseline_clients['total_up'] + baseline_clients['total_down'])
)

baseline_clients.to_csv('baseline_internal_clients.csv', index=False)
print('=== Baseline interno — clientes ===')
display(baseline_clients.head())

# ── 2) Baseline de intervalos temporais internos ──────────────
int_train_sorted = internal_train.sort_values(['src_ip', 'timestamp'])

def compute_intervals(df):
    df = df.copy()
    df['delta'] = df['timestamp'].diff()
    return df[df['delta'].notna()]

train_with_delta = int_train_sorted.groupby('src_ip', group_keys=False).apply(compute_intervals)

baseline_intervals = train_with_delta.groupby('src_ip').agg(
    avg_interval = ('delta', 'mean'),
    std_interval = ('delta', 'std'),
    min_interval = ('delta', 'min'),
    n_flows      = ('delta', 'count')
).reset_index()

baseline_intervals['cv_interval'] = (
    baseline_intervals['std_interval'] / baseline_intervals['avg_interval']
)

baseline_intervals.to_csv('baseline_internal_intervals.csv', index=False)
print('\n=== Baseline interno — intervalos ===')
display(baseline_intervals.head())

# ── 3) Baseline DNS (porta 53) ────────────────────────────────
dns_train = internal_train[internal_train['port'] == 53].copy()

dns_full = dns_train.groupby('src_ip').agg(
    dns_flows = ('up_bytes', 'count')
).reset_index()

print('\n=== Baseline DNS por IP interno ===')
display(dns_full.describe())

# Threshold DNS (R3)
mean_dns = dns_full['dns_flows'].mean()
std_dns  = dns_full['dns_flows'].std()
THRESHOLD_DNS_FLOWS = mean_dns + 3 * std_dns
print(f'\nTHRESHOLD_DNS_FLOWS (μ+3σ): {THRESHOLD_DNS_FLOWS:.2f}')

# ── 4) Baseline por país de destino ───────────────────────────
int_ext = internal_train[~internal_train['dst_ip'].apply(is_private)].copy()
int_ext['dst_cc'] = int_ext['dst_ip'].apply(get_country)

country_flows = int_ext.groupby('dst_cc').agg(count=('dst_ip','count')).reset_index()
country_flows = country_flows.sort_values('count', ascending=False)
country_flows.to_csv('baseline_internal_country_flows.csv', index=False)

country_up = int_ext.groupby('dst_cc').agg(up_bytes=('up_bytes','sum')).reset_index()
country_up = country_up.sort_values('up_bytes', ascending=False)
country_up.to_csv('baseline_internal_country_up.csv', index=False)

FLOWS_MIN_FREQ = 100
paises_freq  = set(country_flows[country_flows['count'] >= FLOWS_MIN_FREQ]['dst_cc'])
paises_raros = set(country_flows[country_flows['count'] <  FLOWS_MIN_FREQ]['dst_cc'])

print('\n=== Baseline por país ===')
display(country_flows)
print(f'\nPaíses frequentes (>= {FLOWS_MIN_FREQ}): {sorted(paises_freq)}')
print(f'Países raros     (<  {FLOWS_MIN_FREQ}): {sorted(paises_raros)}')

=== Baseline interno — clientes ===


,src_ip,n_flows,total_up,total_down,n_dst_servers,n_ports,ratio_up_down
0,192.168.110.100,9056,91068958,833452725,207,2,0.098504
1,192.168.110.101,7763,78887542,731584106,180,2,0.097335
2,192.168.110.102,8840,87834921,813138701,206,2,0.097489
3,192.168.110.103,2925,29927189,265421411,83,2,0.101328
4,192.168.110.104,9517,96774323,870030285,203,2,0.100097



=== Baseline interno — intervalos ===


,src_ip,avg_interval,std_interval,min_interval,n_flows,cv_interval
0,192.168.110.100,431.807951,1808.104860,0.0,9055,4.187289
1,192.168.110.101,532.820407,5532.551999,0.0,7762,10.383521
2,192.168.110.102,446.928159,1884.619485,0.0,8839,4.216829
3,192.168.110.103,430.090971,1823.647016,0.0,2924,4.240143
4,192.168.110.104,488.305591,4964.176560,0.0,9516,10.166127



=== Baseline DNS por IP interno ===


,dns_flows
count,195.000000
mean,589.435897
std,292.739249
min,35.000000
25%,390.000000
50%,571.000000
75%,754.000000
max,1581.000000



THRESHOLD_DNS_FLOWS (μ+3σ): 1467.65

=== Baseline por país ===


,dst_cc,count
27,PT,254803
34,US,180203
6,CA,100397
12,FR,71362
24,NL,39070
13,GB,8924
11,ES,8158
16,IE,7170
5,BR,6400
18,IN,480



Países frequentes (>= 100): ['AU', 'BR', 'CA', 'DE', 'ES', 'FR', 'GB', 'HK', 'IE', 'IN', 'IT', 'JP', 'NL', 'PT', 'SE', 'SG', 'US']
Países raros     (<  100): ['AE', 'AT', 'BE', 'BH', 'CH', 'CL', 'CN', 'ID', 'IL', 'KR', 'MX', 'MY', 'NO', 'PH', 'RO', 'SA', 'TH', 'TW', 'ZA']


In [7]:
# Task 2 - Regras UEBA

#A primeira regra incide sobre a deteção de beaconing/BotNet interno, 
#assumindo que um host comprometido tende a comunicar com maior frequeência temporal
#do que um utilizador legítimo.

In [8]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 5 — Task 1: Baselines externos
# Agrega métricas por IP externo (volume, ratio, intervals)
# ═══════════════════════════════════════════════════════════════

# ── 1) Baseline de clientes externos ──────────────────────────
ext_clients = external_train.groupby('src_ip').agg(
    n_flows      = ('up_bytes',   'count'),
    total_up     = ('up_bytes',   'sum'),
    total_down   = ('down_bytes', 'sum'),
    n_dst_servers= ('dst_ip',     'nunique'),
    n_ports      = ('port',       'nunique')
).reset_index()

ext_clients['ratio_up_down'] = (
    ext_clients['total_up'] /
    (ext_clients['total_up'] + ext_clients['total_down'])
)

ext_clients.to_csv('baseline_external_clients.csv', index=False)
print('=== Baseline externo — clientes ===')
display(ext_clients.head())
display(ext_clients[['n_flows', 'ratio_up_down']].describe())

# ── 2) Baseline de intervalos externos ────────────────────────
ext_train_sorted = external_train.sort_values(['src_ip', 'timestamp'])

ext_with_delta = ext_train_sorted.groupby('src_ip', group_keys=False).apply(compute_intervals)

ext_intervals = ext_with_delta.groupby('src_ip').agg(
    avg_interval = ('delta', 'mean'),
    std_interval = ('delta', 'std'),
    min_interval = ('delta', 'min'),
    n_flows      = ('delta', 'count')
).reset_index()

ext_intervals['cv_interval'] = (
    ext_intervals['std_interval'] / ext_intervals['avg_interval']
)

ext_intervals.to_csv('baseline_external_intervals.csv', index=False)
print('\n=== Baseline externo — intervalos ===')
display(ext_intervals.head())
display(ext_intervals[['avg_interval', 'std_interval', 'cv_interval']].describe())

=== Baseline externo — clientes ===


,src_ip,n_flows,total_up,total_down,n_dst_servers,n_ports,ratio_up_down
0,188.83.81.100,4231,47531169,405389881,1,1,0.104944
1,188.83.81.101,6871,78167671,666194675,1,1,0.105013
2,188.83.81.102,2847,31582173,268313175,1,1,0.105311
3,188.83.81.103,5433,61672108,524257914,1,1,0.105255
4,188.83.81.104,4078,45608246,387331753,1,1,0.105345


,n_flows,ratio_up_down
count,194.000000,194.000000
mean,3535.876289,0.105214
std,1781.682768,0.000570
min,34.000000,0.100912
25%,2211.250000,0.104937
50%,3507.000000,0.105192
75%,4666.750000,0.105509
max,9596.000000,0.107521



=== Baseline externo — intervalos ===


,src_ip,avg_interval,std_interval,min_interval,n_flows,cv_interval
0,188.83.81.100,581.105437,2134.130330,0.0,4230,3.672535
1,188.83.81.101,646.172926,4296.857447,0.0,6870,6.649702
2,188.83.81.102,1043.152143,16517.845788,0.0,2846,15.834551
3,188.83.81.103,811.507732,19111.468665,0.0,5432,23.550569
4,188.83.81.104,773.214128,11980.636285,0.0,4077,15.494591


,avg_interval,std_interval,cv_interval
count,194.000000,194.000000,194.000000
mean,834.494356,10436.733429,11.289532
std,298.077902,7399.971825,5.469130
min,172.848485,413.539230,2.392496
25%,625.881193,3774.771377,6.099408
50%,770.761181,9479.604833,12.038582
75%,944.738895,14942.547500,15.489386
max,2449.470769,36958.219045,23.550569


In [9]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 6 — Regra R1: BotNet Interna
# Objetivo: detetar hosts internos comprometidos, combinando DUAS
# perspetivas complementares de deteção de atividade de botnet:
#
#   A) Comportamento do HOST — fan-out elevado e tráfego temporal
#      demasiado regular (padrão clássico de beaconing).
#   B) Comportamento do PAR (src_ip, dst_ip) interno — comunicações
#      NOVAS nunca vistas no baseline, ou comunicações já existentes
#      com aumento significativo de tráfego.
#
# Nota importante: destinos internos "populares" (servidores
# partilhados por muitos clientes, ex.: file/backup/update server)
# são EXCLUÍDOS do cálculo dos thresholds de B, pois tráfego elevado
# para esses servidores é comportamento normal, não anómalo.
#
# Nota sobre este dataset: TODOS os pares interno-interno do baseline
# têm como destino um servidor popular. Não sobra nenhum par "normal"
# para calibrar um threshold de volume, por isso essa condição fica
# desativada (threshold = +inf) e a sub-regra B passa a assentar
# apenas na deteção de comunicações NOVAS entre hosts internos.
#
# Um host é marcado como suspeito por R1 se disparar A OU B.
# ═══════════════════════════════════════════════════════════════

# ── A) Sub-regra por HOST: fan-out + regularidade temporal ────────
df_r1_host = baseline_clients[['src_ip', 'n_flows', 'n_dst_servers']].merge(
    baseline_intervals[['src_ip', 'cv_interval']],
    on='src_ip', how='inner'
)

mean_dst = df_r1_host['n_dst_servers'].mean()
std_dst  = df_r1_host['n_dst_servers'].std()
THRESHOLD_DST_R1 = mean_dst + 3 * std_dst

mean_cv = df_r1_host['cv_interval'].mean()
std_cv  = df_r1_host['cv_interval'].std()
THRESHOLD_CV_LOW_R1 = mean_cv - 1 * std_cv

df_r1_host['suspeito_host'] = (
    (df_r1_host['n_dst_servers'] > THRESHOLD_DST_R1) &
    (df_r1_host['cv_interval']   < THRESHOLD_CV_LOW_R1)
)

print('=== R1.A — Thresholds (comportamento do host) ===')
print(f'THRESHOLD_DST_R1       : {THRESHOLD_DST_R1:.2f}  (mean={mean_dst:.2f}, std={std_dst:.2f})')
print(f'THRESHOLD_CV_LOW_R1    : {THRESHOLD_CV_LOW_R1:.2f}  (mean={mean_cv:.2f}, std={std_cv:.2f})')
print(f'IPs suspeitos por R1.A no baseline: {df_r1_host["suspeito_host"].sum()}')

suspeitos_r1_host_baseline = df_r1_host[df_r1_host['suspeito_host']].copy()
if len(suspeitos_r1_host_baseline) > 0:
    display(suspeitos_r1_host_baseline.sort_values('n_dst_servers', ascending=False).head(10))
else:
    print('OK: nenhum IP do baseline dispara R1.A.')

# ── B) Sub-regra por PAR interno: novo ou aumento de tráfego ──────
int_int_train = internal_train[
    internal_train['dst_ip'].apply(is_private)
].copy()

baseline_pairs = int_int_train.groupby(['src_ip', 'dst_ip']).agg(
    n_flows_pair     = ('up_bytes', 'count'),
    total_bytes_pair = ('up_bytes', 'sum')
).reset_index()

# Identificar destinos internos "populares" (servidores partilhados por
# muitos clientes distintos) para EXCLUIR do cálculo do threshold de B.
THRESHOLD_POPULARIDADE_R1 = 10
dst_popularity = baseline_pairs.groupby('dst_ip')['src_ip'].nunique().reset_index(name='n_clients')
servidores_internos_r1 = set(
    dst_popularity[dst_popularity['n_clients'] > THRESHOLD_POPULARIDADE_R1]['dst_ip']
)

print(f'\nServidores internos identificados (destinos populares, > {THRESHOLD_POPULARIDADE_R1} clientes): '
      f'{sorted(servidores_internos_r1)}')

baseline_pairs_sem_servidores = baseline_pairs[
    ~baseline_pairs['dst_ip'].isin(servidores_internos_r1)
].copy()

# Se não sobrar nenhum par "normal" (não-servidor) no baseline, não há dados
# para calibrar um threshold de volume: desativa-se essa condição (+inf) em
# vez de propagar um NaN, e a sub-regra B passa a depender só de novidade.
if len(baseline_pairs_sem_servidores) > 0:
    mean_bytes_pair = baseline_pairs_sem_servidores['total_bytes_pair'].mean()
    std_bytes_pair  = baseline_pairs_sem_servidores['total_bytes_pair'].std()
    THRESHOLD_BYTES_PAIR_R1 = mean_bytes_pair + 3 * std_bytes_pair
else:
    mean_bytes_pair = float('nan')
    std_bytes_pair  = float('nan')
    THRESHOLD_BYTES_PAIR_R1 = float('inf')
    print('\nAviso: todo o tráfego interno-interno do baseline tem como destino um '
          'servidor popular. Não existem pares "normais" para calibrar o threshold '
          'de volume — a condição de "aumento significativo" fica desativada '
          '(threshold = +inf) e a sub-regra B assenta apenas na deteção de '
          'comunicações NOVAS entre hosts internos.')

# Pares conhecidos para deteção de comunicações NOVAS: incluem-se todos os
# pares do baseline (mesmo os servidores populares), pois um par já visto
# com um servidor popular não deve ser considerado "novo" no teste.
pares_conhecidos_r1 = set(zip(baseline_pairs['src_ip'], baseline_pairs['dst_ip']))

baseline_pairs.to_csv('baseline_internal_pairs.csv', index=False)

print('\n=== R1.B — Thresholds (comportamento do par interno, sem servidores populares) ===')
print(f'Total de pares internos distintos no baseline         : {len(baseline_pairs)}')
print(f'Pares após excluir servidores populares (p/ threshold) : {len(baseline_pairs_sem_servidores)}')
if len(baseline_pairs_sem_servidores) > 0:
    print(f'THRESHOLD_BYTES_PAIR_R1: {THRESHOLD_BYTES_PAIR_R1:.2f}  (mean={mean_bytes_pair:.2f}, std={std_bytes_pair:.2f})')
else:
    print(f'THRESHOLD_BYTES_PAIR_R1: {THRESHOLD_BYTES_PAIR_R1} (condição de volume desativada)')

suspeitos_r1_pairs_baseline = baseline_pairs_sem_servidores[
    baseline_pairs_sem_servidores['total_bytes_pair'] > THRESHOLD_BYTES_PAIR_R1
].copy()
print(f'Pares (não-servidor) com volume > threshold no baseline: {len(suspeitos_r1_pairs_baseline)}')
if len(suspeitos_r1_pairs_baseline) > 0:
    display(suspeitos_r1_pairs_baseline.sort_values('total_bytes_pair', ascending=False).head(10))
else:
    print('OK: nenhum par (não-servidor) do baseline dispara R1.B por volume.')

# ── Validação combinada no baseline ───────────────────────────────
print(f'\n=== R1 — Validação combinada no baseline ===')
print(f'IPs suspeitos (A: comportamento do host)  : {len(suspeitos_r1_host_baseline)}')
print(f'Pares suspeitos (B: comportamento do par) : {len(suspeitos_r1_pairs_baseline)}')

print('\nRegra R1 (para aplicar no TEST) — suspeito se A OU B:')
print('  A) n_dst_servers > THRESHOLD_DST_R1  E  cv_interval < THRESHOLD_CV_LOW_R1')
print('  B) dst_ip não é um servidor popular conhecido, E:')
print('       (src_ip, dst_ip) não está em pares_conhecidos_r1  [comunicação NOVA]')
print('       OU total_bytes_pair > THRESHOLD_BYTES_PAIR_R1     [AUMENTO significativo]')

=== R1.A — Thresholds (comportamento do host) ===
THRESHOLD_DST_R1       : 270.99  (mean=129.94, std=47.01)
THRESHOLD_CV_LOW_R1    : 5.76  (mean=11.92, std=6.16)
IPs suspeitos por R1.A no baseline: 1


,src_ip,n_flows,n_dst_servers,cv_interval,suspeito_host
115,192.168.110.206,11820,271,4.157089,True



Servidores internos identificados (destinos populares, > 10 clientes): ['192.168.110.222', '192.168.110.230', '192.168.110.238']

Aviso: todo o tráfego interno-interno do baseline tem como destino um servidor popular. Não existem pares "normais" para calibrar o threshold de volume — a condição de "aumento significativo" fica desativada (threshold = +inf) e a sub-regra B assenta apenas na deteção de comunicações NOVAS entre hosts internos.

=== R1.B — Thresholds (comportamento do par interno, sem servidores populares) ===
Total de pares internos distintos no baseline         : 585
Pares após excluir servidores populares (p/ threshold) : 0
THRESHOLD_BYTES_PAIR_R1: inf (condição de volume desativada)
Pares (não-servidor) com volume > threshold no baseline: 0
OK: nenhum par (não-servidor) do baseline dispara R1.B por volume.

=== R1 — Validação combinada no baseline ===
IPs suspeitos (A: comportamento do host)  : 1
Pares suspeitos (B: comportamento do par) : 0

Regra R1 (para aplicar no T

In [10]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 6b — Regra R1 aplicada ao TEST (A + B combinadas)
# B exclui destinos que são servidores internos populares (definidos
# no baseline), para evitar falsos positivos em tráfego normal.
# Inclui também o detalhe, por IP, de qual sinal (A, B, ou ambos)
# disparou a suspeita.
# ═══════════════════════════════════════════════════════════════

internal_test = pd.read_json(INTERNAL_TEST)
print(f'internal_test10: {len(internal_test):,} flows')

# ── A) Comportamento do host no TEST ──────────────────────────────
test_clients = internal_test.groupby('src_ip').agg(
    n_flows      = ('up_bytes', 'count'),
    n_dst_servers= ('dst_ip',   'nunique')
).reset_index()

test_sorted = internal_test.sort_values(['src_ip', 'timestamp'])
test_with_delta = test_sorted.groupby('src_ip', group_keys=False).apply(compute_intervals)
test_intervals = test_with_delta.groupby('src_ip').agg(
    avg_interval=('delta', 'mean'),
    std_interval=('delta', 'std')
).reset_index()
test_intervals['cv_interval'] = test_intervals['std_interval'] / test_intervals['avg_interval']

df_r1_host_test = test_clients.merge(test_intervals[['src_ip', 'cv_interval']], on='src_ip', how='inner')
df_r1_host_test['suspeito_host'] = (
    (df_r1_host_test['n_dst_servers'] > THRESHOLD_DST_R1) &
    (df_r1_host_test['cv_interval']   < THRESHOLD_CV_LOW_R1)
)

# ── B) Comportamento do par no TEST (excluindo servidores populares) ──
int_int_test = internal_test[internal_test['dst_ip'].apply(is_private)].copy()

# Remove logo os fluxos cujo destino é um servidor popular conhecido do baseline
int_int_test_sem_servidores = int_int_test[
    ~int_int_test['dst_ip'].isin(servidores_internos_r1)
].copy()

test_pairs = int_int_test_sem_servidores.groupby(['src_ip', 'dst_ip']).agg(
    n_flows_pair     = ('up_bytes', 'count'),
    total_bytes_pair = ('up_bytes', 'sum')
).reset_index()
test_pairs['par'] = list(zip(test_pairs['src_ip'], test_pairs['dst_ip']))
test_pairs['comunicacao_nova']      = ~test_pairs['par'].isin(pares_conhecidos_r1)
test_pairs['aumento_significativo'] = test_pairs['total_bytes_pair'] > THRESHOLD_BYTES_PAIR_R1
test_pairs['suspeito_par'] = test_pairs['comunicacao_nova'] | test_pairs['aumento_significativo']

# ── Combinação final: host suspeito por A, ou aparece num par suspeito por B ──
srcips_suspeitos_A = set(df_r1_host_test.loc[df_r1_host_test['suspeito_host'], 'src_ip'])
srcips_suspeitos_B = set(test_pairs.loc[test_pairs['suspeito_par'], 'src_ip'])
srcips_suspeitos_r1 = srcips_suspeitos_A | srcips_suspeitos_B

print('=== R1 — Resultados no TEST ===')
print(f'Servidores populares excluídos de B      : {sorted(servidores_internos_r1)}')
print(f'IPs suspeitos por A (host)               : {len(srcips_suspeitos_A)}')
print(f'IPs suspeitos por B (par, sem servidores): {len(srcips_suspeitos_B)}')
print(f'Total de IPs suspeitos (A ou B)          : {len(srcips_suspeitos_r1)}')
print(f'\nIPs internos identificados como suspeitos pela Regra R1:')
for ip in sorted(srcips_suspeitos_r1):
    print(f'  - {ip}')

# ── Detalhe: qual sinal (A, B, ou ambos) disparou por IP ──────────
detalhe_r1 = []
for ip in sorted(srcips_suspeitos_r1):
    disparou_A = ip in srcips_suspeitos_A
    disparou_B = ip in srcips_suspeitos_B
    if disparou_A and disparou_B:
        motivo = 'A e B'
    elif disparou_A:
        motivo = 'A (host: fan-out + regularidade)'
    else:
        motivo = 'B (par: comunicação nova/aumento)'
    detalhe_r1.append({'src_ip': ip, 'disparou_A': disparou_A, 'disparou_B': disparou_B, 'motivo': motivo})

df_detalhe_r1 = pd.DataFrame(detalhe_r1)
print('\n=== R1 — Detalhe por IP suspeito ===')
display(df_detalhe_r1)

df_detalhe_r1.to_csv('r1_detalhe_suspeitos.csv', index=False)

internal_test10: 1,125,194 flows
=== R1 — Resultados no TEST ===
Servidores populares excluídos de B      : ['192.168.110.222', '192.168.110.230', '192.168.110.238']
IPs suspeitos por A (host)               : 2
IPs suspeitos por B (par, sem servidores): 3
Total de IPs suspeitos (A ou B)          : 5

IPs internos identificados como suspeitos pela Regra R1:
  - 192.168.110.116
  - 192.168.110.152
  - 192.168.110.57
  - 192.168.110.59
  - 192.168.110.96

=== R1 — Detalhe por IP suspeito ===


,src_ip,disparou_A,disparou_B,motivo
0,192.168.110.116,True,False,A (host: fan-out + regularidade)
1,192.168.110.152,False,True,B (par: comunicação nova/aumento)
2,192.168.110.57,False,True,B (par: comunicação nova/aumento)
3,192.168.110.59,True,False,A (host: fan-out + regularidade)
4,192.168.110.96,False,True,B (par: comunicação nova/aumento)


In [11]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 7 — Regra R2: Exfiltração via HTTPS (porta 443)
# Objetivo: detetar hosts internos com upload anómalo em HTTPS.
# Métricas:
#   - total_up_443 (bytes enviados em port 443)
#   - ratio_up_443 (fração de upload em relação ao total 443)
# ═══════════════════════════════════════════════════════════════

train_443 = internal_train[internal_train['port'] == 443].copy()
print(f'Flows HTTPS no treino: {len(train_443)}')

baseline_443 = train_443.groupby('src_ip').agg(
    total_up_443   = ('up_bytes',  'sum'),
    total_down_443 = ('down_bytes','sum'),
    n_flows_443    = ('up_bytes',  'count')
).reset_index()

baseline_443['total_bytes_443'] = baseline_443['total_up_443'] + baseline_443['total_down_443']
baseline_443['ratio_up_443']    = baseline_443['total_up_443'] / baseline_443['total_bytes_443']

mean_up_443    = baseline_443['total_up_443'].mean()
std_up_443     = baseline_443['total_up_443'].std()
THRESHOLD_UP_443 = mean_up_443 + 3 * std_up_443

mean_ratio_443  = baseline_443['ratio_up_443'].mean()
std_ratio_443   = baseline_443['ratio_up_443'].std()
THRESHOLD_RATIO_443 = mean_ratio_443 + 3 * std_ratio_443

print('\n=== R2 — Thresholds ===')
print(f'THRESHOLD_UP_443       : {THRESHOLD_UP_443:.2f}  (mean={mean_up_443:.2f}, std={std_up_443:.2f})')
print(f'THRESHOLD_RATIO_443    : {THRESHOLD_RATIO_443:.6f}  (mean={mean_ratio_443:.6f}, std={std_ratio_443:.6f})')

suspeitos_r2_baseline = baseline_443[
    (baseline_443['total_up_443']  > THRESHOLD_UP_443) |
    (baseline_443['ratio_up_443']  > THRESHOLD_RATIO_443)
].copy()

print(f'\nValidação R2 no baseline → IPs suspeitos: {len(suspeitos_r2_baseline)}')
if len(suspeitos_r2_baseline) > 0:
    display(suspeitos_r2_baseline.sort_values('total_up_443', ascending=False).head(10))
else:
    print('OK: nenhum IP do baseline dispara R2.')

print('\nRegra R2 (para aplicar no TEST):')
print('  Suspeito se: total_up_443  > THRESHOLD_UP_443  OU')
print('               ratio_up_443  > THRESHOLD_RATIO_443')

Flows HTTPS no treino: 848563

=== R2 — Thresholds ===
THRESHOLD_UP_443       : 122101002.72  (mean=49505041.35, std=24198653.79)
THRESHOLD_RATIO_443    : 0.104370  (mean=0.097752, std=0.002206)

Validação R2 no baseline → IPs suspeitos: 2


,src_ip,total_up_443,total_down_443,n_flows_443,total_bytes_443,ratio_up_443
157,192.168.110.60,124091789,1147180712,10929,1271272501,0.097612
20,192.168.110.119,5935638,49408422,509,55344060,0.107250



Regra R2 (para aplicar no TEST):
  Suspeito se: total_up_443  > THRESHOLD_UP_443  OU
               ratio_up_443  > THRESHOLD_RATIO_443


In [12]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 8 — Regra R3: Exfiltração via DNS
# Objetivo: detetar hosts internos com volume de queries DNS
#           anormalmente elevado (possível exfiltração por DNS).
# Métrica:
#   - dns_flows acima de THRESHOLD_DNS_FLOWS
# ═══════════════════════════════════════════════════════════════

dns_train = internal_train[internal_train['port'] == 53].copy()

dns_full = dns_train.groupby('src_ip').agg(
    dns_flows = ('up_bytes', 'count')
).reset_index()

mean_dns = dns_full['dns_flows'].mean()
std_dns  = dns_full['dns_flows'].std()
THRESHOLD_DNS_FLOWS = mean_dns + 3 * std_dns

print('=== R3 — Threshold ===')
print(f'THRESHOLD_DNS_FLOWS    : {THRESHOLD_DNS_FLOWS:.2f}  (mean={mean_dns:.2f}, std={std_dns:.2f})')

suspeitos_r3_baseline = dns_full[dns_full['dns_flows'] > THRESHOLD_DNS_FLOWS].copy()

print(f'\nValidação R3 no baseline → IPs suspeitos: {len(suspeitos_r3_baseline)}')
if len(suspeitos_r3_baseline) > 0:
    display(suspeitos_r3_baseline.sort_values('dns_flows', ascending=False).head(10))
else:
    print('OK: nenhum IP do baseline dispara R3.')

print('\nRegra R3 (para aplicar no TEST):')
print('  Suspeito se: dns_flows > THRESHOLD_DNS_FLOWS')

=== R3 — Threshold ===
THRESHOLD_DNS_FLOWS    : 1467.65  (mean=589.44, std=292.74)

Validação R3 no baseline → IPs suspeitos: 3


,src_ip,dns_flows
157,192.168.110.60,1581
164,192.168.110.67,1521
115,192.168.110.206,1471



Regra R3 (para aplicar no TEST):
  Suspeito se: dns_flows > THRESHOLD_DNS_FLOWS


In [13]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 9 — Regra R4: C&C via DNS
# Objetivo: detetar hosts internos com tráfego DNS demasiado
#           regular (padrão de beaconing de comando e controlo).
# Métrica:
#   - cv_interval_dns (coeficiente de variação dos intervalos DNS)
# ═══════════════════════════════════════════════════════════════

train_dns = internal_train[internal_train['port'] == 53].copy()
train_dns = train_dns.sort_values(['src_ip', 'timestamp'])
print(f'Flows DNS no treino: {len(train_dns)}')

dns_with_delta = train_dns.groupby('src_ip', group_keys=False).apply(compute_intervals)

dns_intervals_baseline = dns_with_delta.groupby('src_ip').agg(
    avg_interval_dns = ('delta', 'mean'),
    std_interval_dns = ('delta', 'std'),
    n_flows_dns      = ('delta', 'count')
).reset_index()

dns_intervals_baseline['cv_interval_dns'] = (
    dns_intervals_baseline['std_interval_dns'] /
    dns_intervals_baseline['avg_interval_dns']
)

mean_cv_dns = dns_intervals_baseline['cv_interval_dns'].mean()
std_cv_dns  = dns_intervals_baseline['cv_interval_dns'].std()
THRESHOLD_CV_DNS_LOW = mean_cv_dns - 1 * std_cv_dns

print('\n=== R4 — Threshold ===')
print(f'THRESHOLD_CV_DNS_LOW   : {THRESHOLD_CV_DNS_LOW:.2f}  (mean={mean_cv_dns:.2f}, std={std_cv_dns:.2f})')

suspeitos_r4_baseline = dns_intervals_baseline[
    dns_intervals_baseline['cv_interval_dns'] < THRESHOLD_CV_DNS_LOW
].copy()

print(f'\nValidação R4 no baseline → IPs suspeitos: {len(suspeitos_r4_baseline)}')
if len(suspeitos_r4_baseline) > 0:
    display(suspeitos_r4_baseline.sort_values('cv_interval_dns', ascending=True).head(10))
else:
    print('OK: nenhum IP do baseline dispara R4.')

print('\nRegra R4 (para aplicar no TEST):')
print('  Suspeito se: cv_interval_dns < THRESHOLD_CV_DNS_LOW')

Flows DNS no treino: 114940

=== R4 — Threshold ===
THRESHOLD_CV_DNS_LOW   : 2.28  (mean=4.30, std=2.02)

Validação R4 no baseline → IPs suspeitos: 53


,src_ip,avg_interval_dns,std_interval_dns,n_flows_dns,cv_interval_dns
20,192.168.110.119,4849.487805,6946.330049,41,1.432384
159,192.168.110.62,3245.539007,5339.364828,141,1.645140
83,192.168.110.176,4569.761364,7560.552401,176,1.654474
131,192.168.110.33,4607.282051,7916.789576,234,1.718321
180,192.168.110.85,3726.758794,6441.734376,597,1.728509
86,192.168.110.18,3649.764497,6361.049607,845,1.742866
43,192.168.110.14,3627.717703,6337.317584,418,1.746916
42,192.168.110.139,3544.466591,6222.367593,883,1.755516
11,192.168.110.110,3695.176044,6490.877989,551,1.756582
138,192.168.110.41,3941.866051,6958.654614,433,1.765320



Regra R4 (para aplicar no TEST):
  Suspeito se: cv_interval_dns < THRESHOLD_CV_DNS_LOW


In [14]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 10 — Regra R5: Destinos Externos Anómalos (países raros)
# Objetivo: detetar hosts internos que comunicam com países
#           raros ou nunca vistos no baseline.
# Métrica:
#   - dst_cc do fluxo externo vs. paises_freq / paises_raros
# ═══════════════════════════════════════════════════════════════

country_flows = pd.read_csv('baseline_internal_country_flows.csv')

FLOWS_MIN_FREQ = 100
paises_freq  = set(country_flows[country_flows['count'] >= FLOWS_MIN_FREQ]['dst_cc'])
paises_raros = set(country_flows[country_flows['count'] <  FLOWS_MIN_FREQ]['dst_cc'])

print('=== R5 — Baseline por país ===')
display(country_flows)

print(f'\nTotal de países no baseline    : {len(country_flows)}')
print(f'Países frequentes (>= {FLOWS_MIN_FREQ})  : {len(paises_freq)}')
print(f'  {sorted(paises_freq)}')
print(f'Países raros      (<  {FLOWS_MIN_FREQ})  : {len(paises_raros)}')
print(f'  {sorted(paises_raros)}')

print('\nRegra R5 (para aplicar no TEST):')
print('  Suspeito se: dst_cc ∈ paises_raros  OU')
print('               dst_cc não está no baseline (país novo)')

=== R5 — Baseline por país ===


,dst_cc,count
0,PT,254803
1,US,180203
2,CA,100397
3,FR,71362
4,NL,39070
5,GB,8924
6,ES,8158
7,IE,7170
8,BR,6400
9,IN,480



Total de países no baseline    : 36
Países frequentes (>= 100)  : 17
  ['AU', 'BR', 'CA', 'DE', 'ES', 'FR', 'GB', 'HK', 'IE', 'IN', 'IT', 'JP', 'NL', 'PT', 'SE', 'SG', 'US']
Países raros      (<  100)  : 19
  ['AE', 'AT', 'BE', 'BH', 'CH', 'CL', 'CN', 'ID', 'IL', 'KR', 'MX', 'MY', 'NO', 'PH', 'RO', 'SA', 'TH', 'TW', 'ZA']

Regra R5 (para aplicar no TEST):
  Suspeito se: dst_cc ∈ paises_raros  OU
               dst_cc não está no baseline (país novo)


In [15]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 11 — Regra R6: Clientes Externos Anómalos
# Objetivo: detetar clientes externos com padrão temporal
#           anómalo (irregular / não humano).
# Nota: enunciado diz "tip: it is not the amount of traffic"
#       → foco principal no cv_interval (irregularidade temporal)
# Métricas:
#   - cv_interval acima de THRESHOLD_EXT_CV
#   - n_flows acima de THRESHOLD_EXT_NFLOWS (condição secundária)
# ═══════════════════════════════════════════════════════════════

ext_clients   = pd.read_csv('baseline_external_clients.csv')
ext_intervals = pd.read_csv('baseline_external_intervals.csv')

# Threshold n_flows externos
mean_nflows_ext = ext_clients['n_flows'].mean()
std_nflows_ext  = ext_clients['n_flows'].std()
THRESHOLD_EXT_NFLOWS = mean_nflows_ext + 3 * std_nflows_ext

# Threshold cv_interval externos (condição PRINCIPAL)
mean_cv_ext = ext_intervals['cv_interval'].mean()
std_cv_ext  = ext_intervals['cv_interval'].std()
THRESHOLD_EXT_CV = mean_cv_ext + 3 * std_cv_ext

print('=== R6 — Thresholds ===')
print(f'THRESHOLD_EXT_NFLOWS   : {THRESHOLD_EXT_NFLOWS:.2f}  (mean={mean_nflows_ext:.2f}, std={std_nflows_ext:.2f})')
print(f'THRESHOLD_EXT_CV       : {THRESHOLD_EXT_CV:.2f}  (mean={mean_cv_ext:.2f}, std={std_cv_ext:.2f})')

suspeitos_r6_nflows = ext_clients[ext_clients['n_flows'] > THRESHOLD_EXT_NFLOWS].copy()
suspeitos_r6_cv     = ext_intervals[ext_intervals['cv_interval'] > THRESHOLD_EXT_CV].copy()

print(f'\nValidação R6 no baseline:')
print(f'  IPs com n_flows > THRESHOLD_EXT_NFLOWS  : {len(suspeitos_r6_nflows)}')
print(f'  IPs com cv_interval > THRESHOLD_EXT_CV  : {len(suspeitos_r6_cv)}')

if len(suspeitos_r6_nflows) == 0 and len(suspeitos_r6_cv) == 0:
    print('OK: nenhum cliente externo do baseline dispara R6.')

print('\nRegra R6 (para aplicar no TEST):')
print('  Suspeito se: cv_interval > THRESHOLD_EXT_CV  (condição principal)  OU')
print('               n_flows     > THRESHOLD_EXT_NFLOWS (condição secundária)')

=== R6 — Thresholds ===
THRESHOLD_EXT_NFLOWS   : 8880.92  (mean=3535.88, std=1781.68)
THRESHOLD_EXT_CV       : 27.70  (mean=11.29, std=5.47)

Validação R6 no baseline:
  IPs com n_flows > THRESHOLD_EXT_NFLOWS  : 1
  IPs com cv_interval > THRESHOLD_EXT_CV  : 0

Regra R6 (para aplicar no TEST):
  Suspeito se: cv_interval > THRESHOLD_EXT_CV  (condição principal)  OU
               n_flows     > THRESHOLD_EXT_NFLOWS (condição secundária)
